# 实战案例：基于注意力的图像描述

## 图像描述技术简介

图像描述的关键是生成自然语言描述图像中可以用语言表述的部分。传统的图像描述技术首先通过分析视觉内容来预测给定图像最可能包含的语义信息，并显式的转化为语言标签（通常为单词、短语或其他结构化描述），再基于这些标签生成自然语言描述句子。这类方法均使用以下的管道式结构实现图像描述任务： 

（1）使用计算机视觉技术来对场景进行分类，检测图像中存在的对象，预测它们的属性以及它们之间的关系，识别发生的动作，将它们映射为一些基本的自然语言描述单元，例如单词、短语或其他结构化描述。 

（2）通过自然语言生成技术（例如，模板，n-gram，语法规则等）将这些单词或者短语进行组合，生成自然语言描述句子。

这种管道式方法虽然充分利用了两个领域的现有技术，设计了一套简单可控的解决方案，然而，也存在若干问题：其一，分阶段的方式限制了两个模态数据间的信息交互；其二，这种方法高度依赖于预先定义的场景、对象、属性和动作的封闭语义类集；其三，这种分阶段的模型存在误差累积问题，前面任务的误差在后面阶段会放大；其四，训练误差不能前向传递。

当前主流的图像描述技术大多采用基于编解码框架的方法直接学习图像到文本描述的映射，其核心思想是建模一个以图像为条件的语言模型，计算视觉模式与文本模式的共现概率；其技术基础是深层神经网络对图文两种不同模态数据的通用表示学习能力，可以形成一个端到端的编码解码模型结构。此类方法中所使用的模型可以被端到端地训练，且不需要显示地定义图像和文本之间的桥梁（状态表示），可以有效避免前述管道式方法的问题。

不同的图像描述编解码模型的区别在于其图像编码器和文本解码器所使用的结构的不同。下表列举了深度学习时代常见的图像描述编解码器组合。

| 图像编码器 | 文本解码器 | 
| :----: | :----: | 
| 整体表示 | RNN | 
| 局部表示 | RNN+注意力 | 
| 局部表示+自注意力 | RNN+注意力 | 
| 局部表示+图网络 | RNN+注意力 | 
| 局部表示+Transformer编码器 | Transformer解码器 | 
| 视觉Transformer | Transformer解码模块 | 

接下来，我们将介绍一个图像编码器为CNN网格表示提取器、文本解码器为RNN+注意力的图像描述方法的具体实现。我们的实现大体上是在复现ARCTIC模型，但是在细节上有一些改变，下面的实现过程会对这些改变做具体说明。此外，[链接](https://github.com/sgrvinod/a-PyTorch-Tutorial-to-Image-Captioning)给出了一个更接近原始ARCTIC模型的代码库，非常推荐大家阅读。本节的部分代码也是受到该代码库的启发。

下面，按照读取数据、定义模型、定义损失函数、选择优化方法、选择评估指标和训练模型的次序，来描述该实战案例。

**本作业要求使用PyTorch实现，在CPU上运行。请补全标有 `# TODO` 的代码。**

## 读取数据

我们使用和VSE++相同的数据集flickr8k，其读取数据流程和VSE++完全一致，这里不再赘述。直接使用下面的命令解压即可，这里还需要安装nltk，我们需要利用它的相关库计算BLEU值。

In [ ]:
# 安装依赖
# !pip install torch torchvision nltk pillow matplotlib
# 解压数据
# !unzip -q ./data/data243982/flickr8k.zip -d ./data/

### 整理数据集

数据集下载完成后，我们需要对其进行处理，以适合之后构造的数据集类读取。对于文本描述，我们首先构建词典，然后根据词典将文本描述转化为向量。对于图像，我们这里仅记录文件路径。如果机器的内存和硬盘空间就比较大，这里也可以将图片读取并处理成三维数组，这样在模型训练和测试的阶段，就不需要再直接读取图片。下面是整理数据集的函数的代码。

In [ ]:
%matplotlib inline
import os
from os.path import join as pjoin
import json
import random
import numpy as np
from collections import defaultdict, Counter
from PIL import Image
from matplotlib import pyplot as plt
from argparse import Namespace

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device('cpu')
print(f'使用设备: {device}')

def create_dataset(data_dir='./data',
                   dataset='flickr8k',
                   captions_per_image=5,
                   min_word_count=5,
                   max_len=30):
    """
    参数：
        data_dir: 数据存储目录
        dataset：数据集名称
        captions_per_image：每张图片对应的文本描述数
        min_word_count：仅考虑在数据集中（除测试集外）出现5次的词
        max_len：文本描述包含的最大单词数
    输出：
        vocab.json, train_data.json, val_data.json, test_data.json
    """
    karpathy_json_path = pjoin(data_dir, '%s/dataset_flickr8k.json' % dataset)
    image_folder = pjoin(data_dir, '%s/images' % dataset)
    output_folder = pjoin(data_dir, '%s' % dataset)

    with open(karpathy_json_path, 'r') as j:
        data = json.load(j)

    image_paths = defaultdict(list)
    image_captions = defaultdict(list)
    vocab = Counter()

    for img in data['images']:
        split = img['split']
        captions = []
        for c in img['sentences']:
            if split != 'test':
                vocab.update(c['tokens'])
            if len(c['tokens']) <= max_len:
                captions.append(c['tokens'])
        if len(captions) == 0:
            continue
        path = os.path.join(image_folder, img['filename'])
        image_paths[split].append(path)
        image_captions[split].append(captions)

    words = [w for w in vocab.keys() if vocab[w] > min_word_count]
    vocab = {k: v + 1 for v, k in enumerate(words)}
    vocab['<pad>'] = 0
    vocab['<unk>'] = len(vocab)
    vocab['<start>'] = len(vocab)
    vocab['<end>'] = len(vocab)

    with open(os.path.join(output_folder, 'vocab.json'), 'w') as fw:
        json.dump(vocab, fw)

    for split in image_paths:
        imgpaths = image_paths[split]
        imcaps = image_captions[split]
        enc_captions = []
        for i, path in enumerate(imgpaths):
            img = Image.open(path)
            if len(imcaps[i]) < captions_per_image:
                captions = imcaps[i] + \
                    [random.choice(imcaps[i]) for _ in range(captions_per_image - len(imcaps[i]))]
            else:
                captions = random.sample(imcaps[i], k=captions_per_image)
            assert len(captions) == captions_per_image
            for j, c in enumerate(captions):
                enc_c = [vocab['<start>']] + [vocab.get(word, vocab['<unk>']) for word in c] + [vocab['<end>']]
                enc_captions.append(enc_c)
        assert len(imgpaths) * captions_per_image == len(enc_captions)
        data = {'IMAGES': imgpaths, 'CAPTIONS': enc_captions}
        with open(pjoin(output_folder, split + '_data.json'), 'w') as fw:
            json.dump(data, fw)

data_dir = './data'
if not os.path.exists(pjoin(data_dir, 'flickr8k', 'vocab.json')):
    create_dataset(data_dir)

我们将介绍一个图像编码器为CNN网格表示提取器、文本解码器为RNN+注意力的图像描述方法的具体实现。我们的实现大体上是在复现ARCTIC模型，但是在细节上有一些改变，下面的实现过程会对这些改变做具体说明。

下面，按照读取数据、定义模型、定义损失函数、选择优化方法、选择评估指标和训练模型的次序，来描述该实战案例。

### 定义数据集类

在准备好的数据集的基础上，我们需要进一步定义PyTorch Dataset类，以使用DataLoader类按批次产生数据。PyTorch中仅预先定义了图像、文本和语音的单模态任务中常见的数据集类。因此，我们需要定义自己的数据集类。

在PyTorch中定义数据集类非常简单，仅需要继承`torch.utils.data.Dataset`类，并实现`__getitem__`和`__len__`两个函数即可。

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, dataset_path, vocab_path, split, captions_per_image=5, max_len=30, transform=None):
        self.split = split
        assert self.split in {'train', 'val', 'test'}
        self.cpi = captions_per_image
        self.max_len = max_len

        with open(dataset_path, 'r') as f:
            self.data = json.load(f)
        with open(vocab_path, 'r') as f:
            self.vocab = json.load(f)

        self.transform = transform
        self.dataset_size = len(self.data['CAPTIONS'])

    def __getitem__(self, i):
        img = Image.open(self.data['IMAGES'][i // self.cpi]).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)

        caplen = len(self.data['CAPTIONS'][i])
        caption = torch.tensor(
            self.data['CAPTIONS'][i] + [self.vocab['<pad>']] * (self.max_len + 2 - caplen),
            dtype=torch.long
        )
        return img, caption, caplen

    def __len__(self):
        return self.dataset_size

### 批量读取数据

利用刚才构造的数据集类，借助DataLoader类构建能够按批次产生训练、验证和测试数据的对象。

In [ ]:
def mktrainval(data_dir, vocab_path, batch_size, workers=0):
    train_tx = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_tx = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_set = ImageTextDataset(os.path.join(data_dir, 'train_data.json'),
                                vocab_path, 'train', transform=train_tx)
    valid_set = ImageTextDataset(os.path.join(data_dir, 'val_data.json'),
                                vocab_path, 'val', transform=val_tx)
    test_set = ImageTextDataset(os.path.join(data_dir, 'test_data.json'),
                                vocab_path, 'test', transform=val_tx)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=workers)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=workers, drop_last=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=workers, drop_last=False)

    return train_loader, valid_loader, test_loader

## 定义模型

ARCTIC模型是一个典型的基于注意力的编解码模型，其编码器为图像网格表示提取器，解码器为循环神经网络。解码器在每生成一个词时，都利用注意力机制考虑当前生成的词和图像中的哪些网格更相关。

### 图像编码器

ARCTIC原始模型使用在ImageNet数据集上预训练过的分类模型VGG19作为图像编码器，VGG19最后一个卷积层作为网格表示提取层。而我们这里使用ResNet-101作为图像编码器，并将其最后一个非全连接层作为网格表示提取层。

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, finetuned=True):
        super(ImageEncoder, self).__init__()
        model = models.resnet101(pretrained=True)
        self.grid_representation_extractor = nn.Sequential(*list(model.children())[:-2])
        for param in self.grid_representation_extractor.parameters():
            param.requires_grad = finetuned

    def forward(self, images):
        out = self.grid_representation_extractor(images)
        return out

### 文本解码器

ARCTIC原始模型使用结合注意力的LSTM作为文本解码器，我们这里使用结合注意力的GRU作为文本解码器，注意力评分函数采用的是加性注意力。下面给出加性注意力和解码器的具体实现。

加性注意力评分函数的具体形式为 $W_2^T{\rm tanh}(W_1 [\mathbf{q}_i; \mathbf{k}_j])$ 。

- 首先将权重 $W_1$ 依照查询q和键k的维度，相应地拆成两组权重，分别将单个查询和一组键映射到到注意力函数隐藏层表示空间；
- 然后将二者相加得到一组维度为attn_dim的表示，并在经过非线性变换后，使用形状为(attn_dim, 1) 的权重 $W_2$ 将其映射为一组数值；
- 再通过softmax函数获取单个查询和所有键的关联程度，即归一化的相关性分数；
- 最后以相关性得分为权重，对值进行加权求和，计算输出特征。这里的值和键是同一组向量表示。

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, query_dim, key_dim, attn_dim):
        super(AdditiveAttention, self).__init__()
        self.attn_w_1_q = nn.Linear(query_dim, attn_dim)
        self.attn_w_1_k = nn.Linear(key_dim, attn_dim)
        self.attn_w_2 = nn.Linear(attn_dim, 1)
        self.tanh = nn.Tanh()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, query, key_value):
        queries = self.attn_w_1_q(query).unsqueeze(1)
        keys = self.attn_w_1_k(key_value)
        attn = self.attn_w_2(self.tanh(queries + keys)).squeeze(2)
        attn = self.softmax(attn)
        output = (attn.unsqueeze(2) * key_value).sum(dim=1)
        return output, attn

解码器前馈过程的实现流程如下：

（1）将图文数据按照文本的实际长度从长到短排序，这是为了使用动态的批大小，以避免\<pad\>参与运算带来的非必要的计算消耗。    

（2）在第一时刻解码前，使用图像表示来初始化GRU的隐状态。

（3）解码的每一时刻的具体操作可以分解为如下4个子操作：
    
- （3.1）获取实际的批大小；

- （3.2）利用GRU前一时刻最后一个隐藏层的状态作为查询，图像表示作为键和值，获取上下文向量；

- （3.3）将上下文向量和当前时刻输入的词表示拼接起来，作为GRU该时刻的输入，获得输出；

- （3.4）使用全连接层和softmax激活函数将GRU的输出映射为词表上的概率分布

In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self, image_code_dim, vocab_size, word_dim, attention_dim, hidden_size, num_layers, dropout=0.5):
        super(AttentionDecoder, self).__init__()
        self.embed = nn.Embedding(vocab_size, word_dim)
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        self.attention = AdditiveAttention(hidden_size, image_code_dim, attention_dim)
        self.init_state = nn.Linear(image_code_dim, num_layers * hidden_size)
        self.rnn = nn.GRU(word_dim + image_code_dim, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size

    def init_hidden_state(self, image_code, captions, cap_lens):
        batch_size, image_code_dim = image_code.shape[0], image_code.shape[1]
        image_code = image_code.permute(0, 2, 3, 1)
        image_code = image_code.view(batch_size, -1, image_code_dim)

        sorted_cap_lens, sorted_cap_indices = torch.sort(torch.tensor(cap_lens, dtype=torch.long), descending=True)
        captions = captions[sorted_cap_indices]
        image_code = image_code[sorted_cap_indices]

        if captions.dim() == 1:
            captions = captions.unsqueeze(0)
        if image_code.dim() == 2:
            image_code = image_code.unsqueeze(0)

        hidden_state = self.init_state(image_code.mean(dim=1))
        hidden_state = hidden_state.view(batch_size, self.num_layers, self.hidden_size).permute(1, 0, 2).contiguous()

        return image_code, captions, sorted_cap_lens, sorted_cap_indices, hidden_state

    def forward_step(self, image_code, curr_cap_embed, hidden_state):
        context, alpha = self.attention(hidden_state[-1], image_code)
        rnn_input = torch.cat([context, curr_cap_embed], dim=1).unsqueeze(1)
        output, hidden_state = self.rnn(rnn_input, hidden_state)
        preds = self.fc(self.dropout(output.squeeze(1)))
        return preds, alpha, hidden_state

    def forward(self, image_code, captions, cap_lens):
        image_code, captions, sorted_cap_lens, sorted_cap_indices, hidden_state = \
            self.init_hidden_state(image_code, captions, cap_lens)
        batch_size = image_code.shape[0]
        lengths = sorted_cap_lens.numpy() - 1

        predictions = torch.zeros(batch_size, lengths[0], self.vocab_size)
        alphas = torch.zeros(batch_size, lengths[0], image_code.shape[1])

        cap_embeds = self.embed(captions)

        for step in range(lengths[0]):
            real_batch_size = np.where(lengths > step)[0].shape[0]
            preds, alpha, hidden_state = self.forward_step(
                image_code[:real_batch_size],
                cap_embeds[:real_batch_size, step, :],
                hidden_state[:, :real_batch_size, :].contiguous()
            )
            predictions[:real_batch_size, step, :] = preds
            alphas[:real_batch_size, step, :] = alpha

        return predictions, alphas, captions, lengths, sorted_cap_indices

### ARCTIC模型

在定义编码器和解码器完成之后，我们就很容易构建图像描述模型ARCTIC了。仅需要在初始化函数时声明编码器和解码器，然后在前馈函数实现里，将编码器的输出和文本描述作为解码器的输入即可。

这里我们额外定义了束搜索采样函数，用于生成句子，以计算BLEU值。下面的代码详细标注了其具体实现。

In [ ]:
class ARCTIC(nn.Module):
    def __init__(self, image_code_dim, vocab, word_dim, attention_dim, hidden_size, num_layers):
        super(ARCTIC, self).__init__()
        self.vocab = vocab
        self.encoder = ImageEncoder()
        self.decoder = AttentionDecoder(image_code_dim, len(vocab), word_dim, attention_dim, hidden_size, num_layers)

    def forward(self, images, captions, cap_lens):
        image_code = self.encoder(images)
        return self.decoder(image_code, captions, cap_lens)

    def generate_by_beamsearch(self, images, beam_k, max_len):
        vocab_size = len(self.vocab)
        image_codes = self.encoder(images)
        texts = []
        device = images.device

        for image_code in image_codes:
            image_code = image_code.unsqueeze(0)
            ic_dim = image_code.shape[1]
            image_code = image_code.permute(0, 2, 3, 1).view(1, -1, ic_dim)

            k = beam_k
            image_code = image_code.expand(k, -1, -1)

            hidden_state = self.decoder.init_state(image_code.mean(dim=1))
            hidden_state = hidden_state.view(k, self.decoder.num_layers, self.decoder.hidden_size)
            hidden_state = hidden_state.permute(1, 0, 2).contiguous()

            input_word = torch.tensor([self.vocab['<start>']] * k, dtype=torch.long).to(device)
            cur_input = self.decoder.embed(input_word)

            top_k_scores = torch.zeros(k, 1).to(device)
            seqs = torch.full((k, 1), self.vocab['<start>'], dtype=torch.long).to(device)

            complete_seqs = []
            complete_seqs_scores = []

            for step in range(max_len):
                preds, alpha, hidden_state = self.decoder.forward_step(
                    image_code[:k], cur_input, hidden_state
                )
                preds = F.log_softmax(preds, dim=1)
                scores = top_k_scores.expand_as(preds) + preds

                if step == 0:
                    top_k_scores, top_k_words = scores[0].topk(k, dim=0)
                else:
                    top_k_scores, top_k_words = scores.view(-1).topk(k, dim=0)

                prev_word_inds = top_k_words // vocab_size
                next_word_inds = top_k_words % vocab_size

                seqs = torch.cat([seqs[prev_word_inds], next_word_inds.unsqueeze(1)], dim=1)

                incomplete_inds = [ind for ind, word in enumerate(next_word_inds)
                                   if word != self.vocab['<end>']]
                complete_inds = list(set(range(len(next_word_inds))) - set(incomplete_inds))

                if len(complete_inds) > 0:
                    complete_seqs.extend(seqs[complete_inds].tolist())
                    complete_seqs_scores.extend(top_k_scores[complete_inds].tolist())

                k -= len(complete_inds)
                if k == 0:
                    break

                seqs = seqs[incomplete_inds]
                hidden_state = hidden_state[:, prev_word_inds[incomplete_inds], :].contiguous()
                image_code = image_code[prev_word_inds[incomplete_inds]]
                top_k_scores = top_k_scores[incomplete_inds].unsqueeze(1)
                cur_input = self.decoder.embed(next_word_inds[incomplete_inds])

            if len(complete_seqs) == 0:
                complete_seqs = seqs.tolist()
                complete_seqs_scores = top_k_scores.squeeze(1).tolist()

            i = complete_seqs_scores.index(max(complete_seqs_scores))
            texts.append(complete_seqs[i])

        return texts

## 定义损失函数

这里采用了最常用的交叉熵损失作为损失函数。由于同一个训练批次里的文本描述的长度不一致，因此，有大量的不需要计算损失的\<pad\>目标。为了避免计算资源的浪费，这里需要获得实际有效的序列。

In [ ]:
class CrossEntropyLoss(nn.Module):
    def __init__(self):
        super(CrossEntropyLoss, self).__init__()
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, predictions, targets, lengths):
        v1 = []
        v2 = []
        for i, length in enumerate(lengths):
            v1.append(predictions[i, :length])
            v2.append(targets[i, 1:length+1])
        v1 = torch.cat(v1, dim=0)
        v2 = torch.cat(v2, dim=0)
        loss = self.loss_fn(v1, v2)
        return loss

## 选择优化方法

这里选用Adam优化算法来更新模型参数，由于数据集较小，训练轮次少，因此，学习速率在训练过程中并不调整。

In [ ]:
def get_optimizer(model, config):
    return torch.optim.Adam(model.parameters(), lr=config.learning_rate)

## 评估指标

这里借助nltk库实现了图像描述中最常用的评估指标BLEU值，需要注意的是，再调用计算BLEU值之前，要先将文本中人工添加的文本开始符、结束符和占位符去掉。

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

def filter_useless_words(sent, filterd_words):
    return [w for w in sent if w not in filterd_words]

def evaluate(data_loader, model, config):
    model.eval()
    references = []
    hypotheses = []
    vocab_idx2word = {idx: word for word, idx in model.vocab.items()}
    filterd_words = set(['<start>', '<end>', '<pad>'])
    cpi = config.captions_per_image

    global_idx = 0
    with torch.no_grad():
        for imgs, caps, caplens in data_loader:
            imgs = imgs.to(device)
            texts = model.generate_by_beamsearch(imgs, config.beam_k, config.max_len)

            for j in range(len(texts)):
                if global_idx % cpi == 0:
                    hypothesis = filter_useless_words(
                        [vocab_idx2word.get(idx, '<unk>') for idx in texts[j]],
                        filterd_words
                    )
                    hypotheses.append(hypothesis)

                    refs = []
                    for k in range(cpi):
                        ref_idx = global_idx + k
                        if ref_idx < len(data_loader.dataset):
                            _, ref_cap, _ = data_loader.dataset[ref_idx]
                            ref = filter_useless_words(
                                [vocab_idx2word.get(w, '<unk>') for w in ref_cap.tolist()],
                                filterd_words
                            )
                            refs.append(ref)
                    references.append(refs)
                global_idx += 1

    model.train()
    bleu4 = corpus_bleu(references, hypotheses)
    return bleu4

## 训练模型

训练模型过程仍然是分为读取数据、前馈计算、计算损失、更新参数、选择模型五个步骤。

模型训练的具体方案为一共训练10轮，学习速率为0.0005。

In [ ]:
config = Namespace(
    max_len=30,
    captions_per_image=5,
    batch_size=32,
    image_code_dim=2048,
    word_dim=512,
    hidden_size=512,
    attention_dim=512,
    num_layers=1,
    learning_rate=0.0005,
    num_epochs=10,
    grad_clip=5.0,
    alpha_weight=1.0,
    evaluate_step=900,
    checkpoint=None,
    best_checkpoint='model/ARCTIC/best_flickr8k.ckpt',
    last_checkpoint='model/ARCTIC/last_flickr8k.ckpt',
    beam_k=5
)

data_dir = './data'
vocab_path = pjoin(data_dir, 'flickr8k/vocab.json')
train_loader, valid_loader, test_loader = mktrainval(
    pjoin(data_dir, 'flickr8k'), vocab_path, config.batch_size)

with open(vocab_path, 'r') as f:
    vocab = json.load(f)

model = ARCTIC(config.image_code_dim, vocab, config.word_dim,
               config.attention_dim, config.hidden_size, config.num_layers)
model = model.to(device)

optimizer = get_optimizer(model, config)
loss_fn = CrossEntropyLoss()

os.makedirs(os.path.dirname(config.best_checkpoint), exist_ok=True)
model.train()
best_res = 0
print("开始训练")

for epoch in range(config.num_epochs):
    for i, (imgs, caps, caplens) in enumerate(train_loader):
        imgs = imgs.to(device)
        caps = caps.to(device)
        caplens = torch.tensor(caplens, dtype=torch.long)

        optimizer.zero_grad()
        predictions, alphas, sorted_captions, sorted_lengths, sorted_cap_indices = model(imgs, caps, caplens)
        loss = loss_fn(predictions, sorted_captions, sorted_lengths)
        loss += config.alpha_weight * ((1. - alphas.sum(dim=1)) ** 2).mean()
        loss.backward()
        if config.grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()

        if (i + 1) % 100 == 0:
            print(f'epoch {epoch}, step {i+1}: loss={loss.item():.2f}')

        if (i + 1) % config.evaluate_step == 0:
            bleu_score = evaluate(valid_loader, model, config)
            state = {
                'epoch': epoch, 'step': i,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict()
            }
            if best_res < bleu_score:
                best_res = bleu_score
                torch.save(state, config.best_checkpoint)
            torch.save(state, config.last_checkpoint)
            print(f'Validation@epoch {epoch}, step {i+1}, BLEU-4={bleu_score:.4f}')

# 测试
ckpt = torch.load(config.best_checkpoint, map_location=device)
model.load_state_dict(ckpt['model'])
bleu_score = evaluate(test_loader, model, config)
print(f'Test BLEU-4={bleu_score:.4f} (best epoch={ckpt["epoch"]})')